# GAN + VAE Latent Translation — Full Method (Kaggle Notebook)

Trains all three pieces of the method in order:
1. **VAE1** (domain A: real old photos)
2. **VAE2** (domain B: clean photos + synthetic degradation, dual-branch objective)
3. **Latent translation network** (the actual repair mechanism, combining a supervised synthetic-pair loss with adversarial domain alignment against real photos)

Each stage's checkpoint feeds the next -- VAE1 and VAE2 are trained once, then frozen while the translation network trains on top of both.

**Before running anything:**
1. Right sidebar: **Settings > Accelerator > GPU T4 x2**
2. Right sidebar: **Settings > Internet > On**

**Getting your data in:**
- Real old photos: attach your uploaded Kaggle Dataset (see the project's earlier notes on `kaggle datasets create`), or re-scrape with `loc_scraper.py`
- Clean photos: VOC2012, fetched automatically via torchvision
- Damage masks: generated in-notebook via `generate_synthetic_only.py`


## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected — go to Settings (right sidebar) > Accelerator > GPU T4 x2.')


## 2. Install dependencies

In [ ]:
!pip install -q pillow tqdm opencv-python-headless scikit-image scipy pandas
import torch, torchvision
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)


## 3. Recreate the project files

All tested files from the project: VAE architecture (shared by VAE1 and VAE2), the translation network + discriminator, both dataset classes, and all three training scripts.


In [ ]:
import os
os.chdir('/kaggle/working')
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)


In [ ]:
%%writefile models/__init__.py



In [ ]:
%%writefile models/blocks.py
"""
Basic building blocks shared by the VAE encoder and decoder.

These are standard, well-known layer patterns (residual blocks, strided
conv downsampling, transposed-conv upsampling) -- not specific to any one
paper's architecture. The VAE class that assembles them into the actual
"Bringing Old Photos Back to Life"-style domain VAE is in vae.py, and that
assembly (encoder depth, bottleneck design, how mu/logvar are produced) is
the part you're implementing yourself from the paper's description.
"""

import torch
import torch.nn as nn


class ResidualBlock(nn.Module):
    """A standard two-conv residual block with instance normalization.
    Used inside the encoder/decoder to add capacity without changing
    spatial resolution."""

    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3, padding=0),
            nn.InstanceNorm2d(channels, affine=True),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3, padding=0),
            nn.InstanceNorm2d(channels, affine=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.block(x)


class DownsampleBlock(nn.Module):
    """Strided conv that halves spatial resolution and doubles channels
    (up to a cap), used to build the encoder's downsampling path."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpsampleBlock(nn.Module):
    """Transposed conv that doubles spatial resolution, used to build the
    decoder's upsampling path (mirrors DownsampleBlock)."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


In [ ]:
%%writefile models/vae.py
"""
A convolutional VAE with a *spatial* latent bottleneck (a small feature map,
not a single flattened vector) rather than the more familiar
flatten-to-a-vector VAE design.

Why spatial: "Bringing Old Photos Back to Life" needs the latent
representation to preserve rough spatial layout, so that later (in the
mapping network you'll build next) a damage mask can be used to tell the
model *where* in the latent space to focus repair. A flattened-vector
latent would throw that spatial correspondence away.

This same class is used for both VAE1 (domain A: real old photos) and VAE2
(domain B: clean photos) -- you'll instantiate two separate copies with
their own weights, one per domain, trained independently in
train_vae_domain_a.py / train_vae_domain_b.py.
"""

import torch
import torch.nn as nn

from models.blocks import ResidualBlock, DownsampleBlock, UpsampleBlock


class Encoder(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()

        # Initial conv, no downsampling yet
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, base_channels, kernel_size=7, padding=0),
            nn.InstanceNorm2d(base_channels, affine=True),
            nn.ReLU(inplace=True),
        ]

        # Downsampling path: halve spatial resolution each step, double
        # channels up to max_channels
        channels = base_channels
        for _ in range(n_downsample):
            next_channels = min(channels * 2, max_channels)
            layers.append(DownsampleBlock(channels, next_channels))
            channels = next_channels

        # Residual blocks at the bottleneck resolution, adding capacity
        # without further downsampling
        for _ in range(n_residual_blocks):
            layers.append(ResidualBlock(channels))

        self.backbone = nn.Sequential(*layers)

        # Separate 1x1 convs producing the mean and log-variance maps of
        # the latent distribution -- same spatial size as the backbone
        # output, just a different channel count
        self.to_mu = nn.Conv2d(channels, latent_channels, kernel_size=1)
        self.to_logvar = nn.Conv2d(channels, latent_channels, kernel_size=1)

        self.bottleneck_channels = channels

    def forward(self, x: torch.Tensor):
        features = self.backbone(x)
        mu = self.to_mu(features)
        logvar = self.to_logvar(features)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, out_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()

        # Figure out the bottleneck channel count the same way the
        # encoder did, so the shapes line up
        channels = base_channels
        for _ in range(n_downsample):
            channels = min(channels * 2, max_channels)

        layers = [nn.Conv2d(latent_channels, channels, kernel_size=1)]

        for _ in range(n_residual_blocks):
            layers.append(ResidualBlock(channels))

        # Upsampling path: mirror of the encoder's downsampling path
        channel_sequence = []
        c = base_channels
        for _ in range(n_downsample):
            channel_sequence.append(min(c * 2, max_channels))
            c = min(c * 2, max_channels)
        channel_sequence = [base_channels] + channel_sequence
        # channel_sequence e.g. [64, 128, 256, 512] for n_downsample=3;
        # we walk it backwards to go from bottleneck back to base_channels
        for i in range(n_downsample):
            in_ch = channel_sequence[n_downsample - i]
            out_ch = channel_sequence[n_downsample - i - 1]
            layers.append(UpsampleBlock(in_ch, out_ch))

        layers += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(base_channels, out_channels, kernel_size=7, padding=0),
            nn.Tanh(),  # output in [-1, 1], matches how we'll normalize images
        ]

        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class DomainVAE(nn.Module):
    """
    Full VAE: encode -> reparameterize -> decode.

    Instantiate one of these per domain (real old photos / clean photos).
    The `Encoder`/`Decoder` above are shared *class* definitions but each
    DomainVAE instance gets its own independently-trained weights.
    """

    def __init__(self, in_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()
        self.encoder = Encoder(in_channels, base_channels, n_downsample,
                                n_residual_blocks, latent_channels, max_channels)
        self.decoder = Decoder(in_channels, base_channels, n_downsample,
                                n_residual_blocks, latent_channels, max_channels)

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """The standard VAE reparameterization trick: sample z = mu + eps*std
        where eps ~ N(0, 1), so gradients can flow through the sampling step."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar


def vae_loss(recon: torch.Tensor, target: torch.Tensor, mu: torch.Tensor,
             logvar: torch.Tensor, kl_weight: float = 1.0):
    """
    Standard VAE loss = reconstruction term + KL divergence term.

    Reconstruction uses L1 (tends to give sharper results than MSE for
    images -- this is a common choice in image-translation VAEs, not
    something unique to this paper).

    KL divergence pulls the latent distribution toward a standard normal,
    which is what makes the latent space smooth/well-structured enough for
    the mapping network to later translate between domains.
    """
    recon_loss = torch.nn.functional.l1_loss(recon, target)
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + kl_weight * kl_loss
    return total_loss, recon_loss, kl_loss


def vae2_loss(recon_clean, recon_degraded, clean_target, mu_clean, logvar_clean,
              mu_degraded, logvar_degraded, kl_weight: float = 1.0, consistency_weight: float = 1.0):
    """
    VAE2's training objective is different from VAE1's plain reconstruction:
    it needs to learn a latent space where a clean photo AND a synthetically
    degraded version of it land close together, and BOTH decode back to the
    clean image. This is what lets the translation network later map a
    repaired latent through VAE2's decoder and get a clean-looking result.

    Four terms:
      1. Standard reconstruction of the clean branch (clean in -> clean out)
      2. Cross-reconstruction of the degraded branch (degraded in -> CLEAN
         out, not degraded out) -- this is the key difference from a plain
         autoencoder; it directly teaches "decode toward clean" regardless
         of which branch encoded the input
      3. Latent consistency: pulls the degraded branch's latent toward the
         clean branch's latent (clean side detached, so gradient flows
         into fixing the degraded encoder rather than both sides drifting
         together into a degenerate shortcut)
      4. KL divergence on both branches (standard VAE regularization)
    """
    recon_loss_clean = torch.nn.functional.l1_loss(recon_clean, clean_target)
    recon_loss_degraded = torch.nn.functional.l1_loss(recon_degraded, clean_target)

    consistency_loss = torch.nn.functional.l1_loss(mu_degraded, mu_clean.detach())

    kl_clean = -0.5 * torch.mean(1 + logvar_clean - mu_clean.pow(2) - logvar_clean.exp())
    kl_degraded = -0.5 * torch.mean(1 + logvar_degraded - mu_degraded.pow(2) - logvar_degraded.exp())
    kl_loss = kl_clean + kl_degraded

    total_loss = (recon_loss_clean + recon_loss_degraded
                  + consistency_weight * consistency_loss
                  + kl_weight * kl_loss)

    return total_loss, recon_loss_clean, recon_loss_degraded, consistency_loss, kl_loss


In [ ]:
%%writefile models/translation_net.py
"""
The latent translation network -- this is where the actual "repair"
happens. Takes a latent map from either VAE1 (real damaged photo) or VAE2
(synthetic degraded photo), plus a damage mask resized to latent
resolution, and outputs a translated latent that VAE2's decoder can turn
into a clean-looking image.

Also defines the latent-space discriminator used for adversarial training
on real photos, where we have no ground-truth clean version to supervise
against directly (see train_translation_net.py for how the two training
signals -- supervised synthetic pairs + adversarial real photos -- combine).
"""

import torch
import torch.nn as nn

from models.blocks import ResidualBlock


class LatentTranslationNet(nn.Module):
    def __init__(self, latent_channels: int = 64, n_residual_blocks: int = 6):
        super().__init__()

        # Fuse the latent map with the (resized) damage mask -- this is the
        # actual mask-conditioning mechanism: the mask is just concatenated
        # as an extra input channel, giving every residual block access to
        # "is this spatial location damaged" throughout the network.
        self.input_conv = nn.Sequential(
            nn.Conv2d(latent_channels + 1, latent_channels, kernel_size=3, padding=1),
            nn.InstanceNorm2d(latent_channels, affine=True),
            nn.ReLU(inplace=True),
        )

        self.residual_blocks = nn.Sequential(
            *[ResidualBlock(latent_channels) for _ in range(n_residual_blocks)]
        )

        # Output projection back to latent space -- no activation, since
        # latent values (VAE means) aren't bounded to any fixed range
        self.output_conv = nn.Conv2d(latent_channels, latent_channels, kernel_size=3, padding=1)

    def forward(self, latent: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # mask arrives at image resolution; resize to match the latent
        # map's (much smaller) spatial size
        if mask.shape[-2:] != latent.shape[-2:]:
            mask = torch.nn.functional.interpolate(mask, size=latent.shape[-2:], mode="bilinear",
                                                     align_corners=False)
        x = torch.cat([latent, mask], dim=1)
        x = self.input_conv(x)
        x = self.residual_blocks(x)
        return self.output_conv(x)


class LatentDiscriminator(nn.Module):
    """PatchGAN-style discriminator operating directly on latent maps
    (not images). Outputs a spatial grid of real/fake scores rather than
    one global score, which gives a stronger, more localized training
    signal than a single scalar would."""

    def __init__(self, latent_channels: int = 64, base_channels: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(latent_channels, base_channels, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(base_channels * 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=3, padding=1),
            nn.InstanceNorm2d(base_channels * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            # No sigmoid -- used with BCEWithLogitsLoss for numerical stability
            nn.Conv2d(base_channels * 4, 1, kernel_size=3, padding=1),
        )

    def forward(self, latent: torch.Tensor) -> torch.Tensor:
        return self.net(latent)


In [ ]:
%%writefile data/__init__.py



In [ ]:
%%writefile data/real_photo_dataset.py
"""
Dataset loader for VAE1 (domain A) training: real old photos, as collected
by loc_scraper.py / dpla_scraper.py. Unsupervised -- no labels needed, just
a folder of images.
"""

import os
import csv
import random

from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T


class RealOldPhotoDataset(Dataset):
    """
    Expects the folder layout produced by loc_scraper.py / dpla_scraper.py:

        <root>/images/*.jpg
        <root>/manifest.csv   (optional, only used to list valid filenames)

    If manifest.csv is present, filenames are read from it (keeps you in
    sync with whatever passed your download-time validation). Otherwise
    falls back to globbing every image file in <root>/images/.
    """

    def __init__(self, root: str, image_size: int = 256, augment: bool = True):
        self.root = root
        self.images_dir = os.path.join(root, "images")
        self.image_size = image_size
        self.augment = augment

        self.filenames = self._load_filenames()
        if len(self.filenames) == 0:
            raise ValueError(f"No images found under {self.images_dir}")

        # Resize the short side up a bit past image_size so RandomCrop has
        # room to move -- this is a standard cheap augmentation for
        # unsupervised reconstruction training.
        load_size = int(image_size * 1.12)

        transform_list = [
            T.Resize(load_size),
        ]
        if augment:
            transform_list += [
                T.RandomCrop(image_size),
                T.RandomHorizontalFlip(),
            ]
        else:
            transform_list += [T.CenterCrop(image_size)]

        transform_list += [
            T.ToTensor(),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # -> [-1, 1]
        ]
        self.transform = T.Compose(transform_list)

    def _load_filenames(self):
        manifest_path = os.path.join(self.root, "manifest.csv")
        if os.path.exists(manifest_path):
            filenames = []
            with open(manifest_path, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    fn = row.get("local_filename")
                    if fn:
                        filenames.append(fn)
            if filenames:
                return filenames

        # fallback: glob the images directory directly
        if not os.path.isdir(self.images_dir):
            return []
        valid_ext = (".jpg", ".jpeg", ".png")
        return [f for f in os.listdir(self.images_dir) if f.lower().endswith(valid_ext)]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        path = os.path.join(self.images_dir, filename)
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            # If something slipped past validation and is unreadable,
            # fall back to a random other item rather than crashing an
            # entire training run over one bad file.
            return self.__getitem__(random.randrange(len(self)))

        img_tensor = self.transform(img)
        return {"image": img_tensor, "filename": filename}


def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    """Inverse of the Normalize(mean=0.5, std=0.5) above, for saving/viewing
    reconstructed images. Maps [-1, 1] back to [0, 1]."""
    return (tensor * 0.5 + 0.5).clamp(0, 1)


In [ ]:
%%writefile data/vae2_pair_dataset.py
"""
Dataset for VAE2 (domain B) and the latent translation network: yields
(clean, degraded, mask) triples. Degraded images are composited on the fly
using your FilmDamageSimulator masks (screen blend by default -- see
composite_damage.py), same approach as the DiffBIR Stage 1 dataset.

Returning the raw mask (not just the composited degraded image) is what's
new here -- VAE2 training itself doesn't need it, but the translation
network trained on top of VAE2 does, since it uses the mask to know where
to apply heavier repair.
"""

import os
import random

from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
import torchvision.transforms.functional as TF


class VAE2PairDataset(Dataset):
    def __init__(self, clean_dir: str, masks_dir: str, image_size: int = 256, augment: bool = True,
                 blend_mode: str = "screen"):
        if blend_mode not in ("screen", "multiply"):
            raise ValueError(f"blend_mode must be 'screen' or 'multiply', got '{blend_mode}'")
        self.clean_dir = clean_dir
        self.masks_dir = masks_dir
        self.image_size = image_size
        self.augment = augment
        self.blend_mode = blend_mode

        valid_ext = (".jpg", ".jpeg", ".png")
        self.clean_files = [f for f in os.listdir(clean_dir) if f.lower().endswith(valid_ext)]
        self.mask_files = [f for f in os.listdir(masks_dir)
                            if f.lower().endswith(".png") and not f.startswith("binarised_mask")]

        if len(self.clean_files) == 0:
            raise ValueError(f"No clean images found in {clean_dir}")
        if len(self.mask_files) == 0:
            raise ValueError(f"No usable masks found in {masks_dir}")

        load_size = int(image_size * 1.12)
        self.clean_resize = T.Resize(load_size)
        self.image_size_final = image_size

    def __len__(self):
        return len(self.clean_files)

    def _load_clean(self, idx):
        path = os.path.join(self.clean_dir, self.clean_files[idx])
        img = Image.open(path).convert("RGB")
        return self.clean_resize(img)

    def _load_random_mask(self):
        path = os.path.join(self.masks_dir, random.choice(self.mask_files))
        return Image.open(path).convert("L")

    def _synchronized_crop_and_flip(self, clean_img, mask_img):
        mask_img = mask_img.resize(clean_img.size, Image.BILINEAR)

        if self.augment:
            i, j, h, w = T.RandomCrop.get_params(clean_img, output_size=(self.image_size_final, self.image_size_final))
            clean_img = TF.crop(clean_img, i, j, h, w)
            mask_img = TF.crop(mask_img, i, j, h, w)
            if random.random() < 0.5:
                clean_img = TF.hflip(clean_img)
                mask_img = TF.hflip(mask_img)
            if random.random() < 0.5:
                angle = random.choice([90, 180, 270])
                mask_img = TF.rotate(mask_img, angle)
        else:
            clean_img = TF.center_crop(clean_img, (self.image_size_final, self.image_size_final))
            mask_img = TF.center_crop(mask_img, (self.image_size_final, self.image_size_final))

        return clean_img, mask_img

    def __getitem__(self, idx):
        try:
            clean_img = self._load_clean(idx)
            mask_img = self._load_random_mask()
        except Exception:
            return self.__getitem__(random.randrange(len(self)))

        clean_img, mask_img = self._synchronized_crop_and_flip(clean_img, mask_img)

        clean_tensor = TF.to_tensor(clean_img)
        mask_tensor = TF.to_tensor(mask_img)

        if self.blend_mode == "screen":
            degraded_tensor = 1.0 - (1.0 - clean_tensor) * mask_tensor
        else:
            degraded_tensor = clean_tensor * mask_tensor

        normalize = T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        clean_tensor = normalize(clean_tensor)
        degraded_tensor = normalize(degraded_tensor)
        # mask stays in [0, 1] -- it's used as a conditioning signal, not an
        # image to reconstruct, so it doesn't need the [-1, 1] normalization

        return {"clean": clean_tensor, "degraded": degraded_tensor, "mask": mask_tensor}


def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    return (tensor * 0.5 + 0.5).clamp(0, 1)


In [ ]:
%%writefile train_vae_domain_a.py
"""
Train VAE1 (domain A) on real old photos, unsupervised reconstruction.

This is the first, smallest independently-testable piece of the full
pipeline: encoder -> reparameterize -> decoder, trained to reconstruct
real old photos. Once this trains stably, VAE2 (domain B, on clean photos)
uses the exact same architecture/training loop against a different
dataset -- and both feed into the mapping network stage after that.

Usage:
    python train_vae_domain_a.py --data-root ./real_old_photos --epochs 50 \
        --batch-size 8 --image-size 256 --out-dir ./runs/vae_domain_a
"""

import argparse
import os
import time
from datetime import datetime

import torch
from torch.utils.data import DataLoader, RandomSampler
import torchvision.utils as vutils
from tqdm import tqdm

from models.vae import DomainVAE, vae_loss
from data.real_photo_dataset import RealOldPhotoDataset, denormalize


def format_duration(seconds):
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


class Logger:
    """Prints with a wall-clock timestamp AND appends the same line to a log
    file, so you can check progress from outside the live session (e.g.
    tailing the file over SSH, or reopening a Kaggle notebook that's still
    running) rather than only trusting that the visible cell output is
    current. Opens in append mode, so resuming a run continues the same log
    rather than overwriting history."""

    def __init__(self, log_path):
        self.log_path = log_path
        os.makedirs(os.path.dirname(log_path) or ".", exist_ok=True)
        self._file = open(log_path, "a", encoding="utf-8")

    def log(self, msg):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] {msg}"
        print(line, flush=True)
        self._file.write(line + "\n")
        self._file.flush()

    def close(self):
        self._file.close()


def save_reconstruction_grid(model, batch, out_path, device, max_images=8):
    """Saves a side-by-side grid of [original | reconstruction] so you can
    visually sanity-check training progress, not just watch the loss curve."""
    model.eval()
    with torch.no_grad():
        images = batch["image"][:max_images].to(device)
        recon, _, _ = model(images)
        comparison = torch.cat([denormalize(images), denormalize(recon)], dim=0)
        vutils.save_image(comparison, out_path, nrow=max_images)
    model.train()


def main():
    parser = argparse.ArgumentParser(description="Train VAE1 on real old photos (domain A).")
    parser.add_argument("--data-root", type=str, required=True,
                         help="folder containing images/ and manifest.csv from the scraper scripts")
    parser.add_argument("--out-dir", type=str, default="./runs/vae_domain_a")
    parser.add_argument("--image-size", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=8,
                         help="the official repo uses 80-120 across 4 GPUs; start much smaller on one GPU/CPU")
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--kl-weight", type=float, default=0.01,
                         help="weight on the KL term; too high early on can collapse reconstructions to blur")
    parser.add_argument("--latent-channels", type=int, default=64)
    parser.add_argument("--n-downsample", type=int, default=3)
    parser.add_argument("--n-residual-blocks", type=int, default=4)
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--save-every", type=int, default=5, help="save a checkpoint every N epochs")
    parser.add_argument("--sample-every", type=int, default=200,
                         help="save a reconstruction sample grid every N training steps")
    parser.add_argument("--resume", type=str, default=None, help="path to a checkpoint to resume from")
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--amp", action="store_true",
                         help="use automatic mixed precision (fp16) training -- meaningful speedup on modern "
                              "GPUs (T4 and newer) with minimal code cost; no effect on CPU")
    parser.add_argument("--steps-per-epoch", type=int, default=None,
                         help="if set, each 'epoch' samples this many random batches (with replacement) instead "
                              "of iterating the full dataset once -- use this to shorten epoch wall-clock time.")
    parser.add_argument("--log-every", type=int, default=20,
                         help="print a data-loading-vs-compute timing breakdown every N steps. Set to 0 to disable.")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    checkpoints_dir = os.path.join(args.out_dir, "checkpoints")
    samples_dir = os.path.join(args.out_dir, "samples")
    os.makedirs(checkpoints_dir, exist_ok=True)
    os.makedirs(samples_dir, exist_ok=True)

    logger = Logger(os.path.join(args.out_dir, "train_log.txt"))
    log = logger.log

    device = torch.device(args.device)
    log(f"Using device: {device}")
    if device.type == "cuda":
        log(f"  GPU: {torch.cuda.get_device_name(device)}")
    cpu_count = os.cpu_count()
    log(f"  CPUs available: {cpu_count}, --num-workers set to {args.num_workers}")
    if args.num_workers > cpu_count:
        log(f"  Warning: --num-workers ({args.num_workers}) exceeds available CPUs ({cpu_count}); "
            f"this can hurt rather than help. Consider lowering it.")

    log("Building dataset index (scanning images/ and manifest.csv)...")
    dataset = RealOldPhotoDataset(args.data_root, image_size=args.image_size, augment=True)
    log(f"Loaded {len(dataset)} real old photos from {args.data_root}")

    if args.steps_per_epoch:
        num_samples = args.steps_per_epoch * args.batch_size
        sampler = RandomSampler(dataset, replacement=True, num_samples=num_samples)
        dataloader = DataLoader(dataset, batch_size=args.batch_size, sampler=sampler,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))
        log(f"Using --steps-per-epoch {args.steps_per_epoch}: each epoch samples "
            f"{num_samples} images (with replacement) instead of the full {len(dataset)}-image dataset")
    else:
        dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))

    # A small fixed batch, held out purely for visualizing reconstruction
    # quality over time (not used for any gradient updates). This first
    # dataloader fetch is also what spins up worker processes -- can take a
    # few seconds even before any training happens, hence the explicit log.
    log("Fetching a fixed sample batch for visualization (this also starts the dataloader workers)...")
    fixed_batch = next(iter(dataloader))
    log("Dataset ready.")

    model = DomainVAE(
        in_channels=3,
        n_downsample=args.n_downsample,
        n_residual_blocks=args.n_residual_blocks,
        latent_channels=args.latent_channels,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.5, 0.999))

    use_amp = args.amp and device.type == "cuda"
    if args.amp and device.type != "cuda":
        log("Note: --amp has no effect on CPU, ignoring.")
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    start_epoch = 1
    global_step = 0
    if args.resume:
        log(f"Resuming from {args.resume}")
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt.get("global_step", 0)

    num_params = sum(p.numel() for p in model.parameters())
    log(f"Model has {num_params:,} parameters")
    log(f"Starting training: epochs {start_epoch}-{args.epochs}")

    start_time = time.time()

    for epoch in range(start_epoch, args.epochs + 1):
        epoch_start = time.time()
        running_loss, running_recon, running_kl = 0.0, 0.0, 0.0
        running_data_time, running_compute_time = 0.0, 0.0

        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}/{args.epochs}", unit="batch", leave=False)

        batch_end_time = time.time()
        for batch in progress_bar:
            data_time = time.time() - batch_end_time

            compute_start = time.time()
            images = batch["image"].to(device, non_blocking=True)

            optimizer.zero_grad()

            with torch.amp.autocast(device.type, enabled=use_amp):
                recon, mu, logvar = model(images)
                loss, recon_loss, kl_loss = vae_loss(recon, images, mu, logvar, kl_weight=args.kl_weight)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            if device.type == "cuda":
                torch.cuda.synchronize()
            compute_time = time.time() - compute_start

            running_loss += loss.item()
            running_recon += recon_loss.item()
            running_kl += kl_loss.item()
            running_data_time += data_time
            running_compute_time += compute_time
            global_step += 1

            # Live, continuously-updating feedback -- this is what tells you
            # "still working" second by second, rather than waiting on a
            # periodic print that might be minutes away.
            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "data_t": f"{data_time:.2f}s",
                "compute_t": f"{compute_time:.2f}s",
            })

            if args.log_every and global_step % args.log_every == 0:
                msg = (f"  step {global_step}: data_time={data_time:.3f}s compute_time={compute_time:.3f}s "
                       f"({'data-loading-bound' if data_time > compute_time else 'compute-bound'})")
                if device.type == "cuda":
                    mem_alloc = torch.cuda.memory_allocated(device) / 1e9
                    mem_reserved = torch.cuda.memory_reserved(device) / 1e9
                    msg += f" | GPU mem: {mem_alloc:.2f}GB alloc / {mem_reserved:.2f}GB reserved"
                log(msg)

            if global_step % args.sample_every == 0:
                sample_path = os.path.join(samples_dir, f"step_{global_step:07d}.png")
                save_reconstruction_grid(model, fixed_batch, sample_path, device)
                log(f"  Saved sample grid: {sample_path}")

            batch_end_time = time.time()

        n_batches = len(dataloader)
        elapsed = time.time() - start_time
        log(f"[Epoch {epoch}/{args.epochs}] "
            f"loss={running_loss / n_batches:.4f} "
            f"recon={running_recon / n_batches:.4f} "
            f"kl={running_kl / n_batches:.4f} "
            f"avg_data_time={running_data_time / n_batches:.3f}s "
            f"avg_compute_time={running_compute_time / n_batches:.3f}s "
            f"epoch_time={format_duration(time.time() - epoch_start)} "
            f"total_elapsed={format_duration(elapsed)}")

        if epoch % args.save_every == 0 or epoch == args.epochs:
            ckpt_path = os.path.join(checkpoints_dir, f"vae_domain_a_epoch{epoch:04d}.pt")
            torch.save({
                "epoch": epoch,
                "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "args": vars(args),
            }, ckpt_path)
            log(f"  Saved checkpoint: {ckpt_path}")

    total_elapsed = time.time() - start_time
    log(f"Training complete. Total time: {format_duration(total_elapsed)}")
    logger.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_vae_domain_b.py
"""
Train VAE2 (domain B) on clean photos + their synthetically damaged
versions. Unlike VAE1, this trains with a dual-branch objective: both the
clean image AND its degraded version get encoded, and BOTH must decode
back to the CLEAN target, with a consistency term pulling their latents
together. See models/vae.py:vae2_loss for the full reasoning.

Usage:
    python train_vae_domain_b.py --clean-dir ./voc_data --masks-dir ./generated_masks \
        --epochs 50 --batch-size 8 --image-size 256 --out-dir ./runs/vae_domain_b
"""

import argparse
import os
import time
from datetime import datetime

import torch
from torch.utils.data import DataLoader, RandomSampler
import torchvision.utils as vutils
from tqdm import tqdm

from models.vae import DomainVAE, vae2_loss
from data.vae2_pair_dataset import VAE2PairDataset, denormalize


def format_duration(seconds):
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


class Logger:
    def __init__(self, log_path):
        self.log_path = log_path
        os.makedirs(os.path.dirname(log_path) or ".", exist_ok=True)
        self._file = open(log_path, "a", encoding="utf-8")

    def log(self, msg):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] {msg}"
        print(line, flush=True)
        self._file.write(line + "\n")
        self._file.flush()

    def close(self):
        self._file.close()


def save_comparison_grid(model, batch, out_path, device, max_images=6):
    """Saves [clean input | clean recon | degraded input | degraded->clean recon]
    so you can visually confirm both branches are converging toward the
    same clean output, which is the whole point of VAE2's training."""
    model.eval()
    with torch.no_grad():
        n = min(max_images, batch["clean"].shape[0])
        clean = batch["clean"][:n].to(device)
        degraded = batch["degraded"][:n].to(device)

        recon_clean, _, _ = model(clean)
        recon_degraded, _, _ = model(degraded)

        grid = torch.cat([
            denormalize(clean), denormalize(recon_clean),
            denormalize(degraded), denormalize(recon_degraded),
        ], dim=0)
        vutils.save_image(grid, out_path, nrow=n)
    model.train()


def main():
    parser = argparse.ArgumentParser(description="Train VAE2 on clean photos + synthetic degradation (domain B).")
    parser.add_argument("--clean-dir", type=str, required=True, help="folder of clean photos (e.g. VOC2012)")
    parser.add_argument("--masks-dir", type=str, required=True,
                         help="folder of masks from generate_synthetic_only.py")
    parser.add_argument("--blend-mode", type=str, choices=["screen", "multiply"], default="screen")
    parser.add_argument("--out-dir", type=str, default="./runs/vae_domain_b")
    parser.add_argument("--image-size", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--kl-weight", type=float, default=0.01)
    parser.add_argument("--consistency-weight", type=float, default=1.0,
                         help="weight pulling the degraded branch's latent toward the clean branch's latent")
    parser.add_argument("--latent-channels", type=int, default=64)
    parser.add_argument("--n-downsample", type=int, default=3)
    parser.add_argument("--n-residual-blocks", type=int, default=4)
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--save-every", type=int, default=5)
    parser.add_argument("--sample-every", type=int, default=200)
    parser.add_argument("--steps-per-epoch", type=int, default=None)
    parser.add_argument("--log-every", type=int, default=20)
    parser.add_argument("--resume", type=str, default=None)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--amp", action="store_true")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    checkpoints_dir = os.path.join(args.out_dir, "checkpoints")
    samples_dir = os.path.join(args.out_dir, "samples")
    os.makedirs(checkpoints_dir, exist_ok=True)
    os.makedirs(samples_dir, exist_ok=True)

    logger = Logger(os.path.join(args.out_dir, "train_log.txt"))
    log = logger.log

    device = torch.device(args.device)
    log(f"Using device: {device}")
    if device.type == "cuda":
        log(f"  GPU: {torch.cuda.get_device_name(device)}")
    cpu_count = os.cpu_count()
    log(f"  CPUs available: {cpu_count}, --num-workers set to {args.num_workers}")

    log("Building dataset index...")
    dataset = VAE2PairDataset(args.clean_dir, args.masks_dir, image_size=args.image_size,
                               augment=True, blend_mode=args.blend_mode)
    log(f"Loaded {len(dataset)} clean images, {len(dataset.mask_files)} masks "
        f"(blend_mode={args.blend_mode})")

    if args.steps_per_epoch:
        num_samples = args.steps_per_epoch * args.batch_size
        sampler = RandomSampler(dataset, replacement=True, num_samples=num_samples)
        dataloader = DataLoader(dataset, batch_size=args.batch_size, sampler=sampler,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))
        log(f"Using --steps-per-epoch {args.steps_per_epoch}: {num_samples} images/epoch")
    else:
        dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))

    log("Fetching a fixed sample batch for visualization...")
    fixed_batch = next(iter(dataloader))
    log("Dataset ready.")

    model = DomainVAE(
        in_channels=3, n_downsample=args.n_downsample,
        n_residual_blocks=args.n_residual_blocks, latent_channels=args.latent_channels,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.5, 0.999))

    use_amp = args.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    start_epoch = 1
    global_step = 0
    if args.resume:
        log(f"Resuming from {args.resume}")
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt.get("global_step", 0)

    num_params = sum(p.numel() for p in model.parameters())
    log(f"Model has {num_params:,} parameters")
    log(f"Starting training: epochs {start_epoch}-{args.epochs}")

    start_time = time.time()

    for epoch in range(start_epoch, args.epochs + 1):
        epoch_start = time.time()
        running_loss = 0.0
        running_rc, running_rd, running_cons, running_kl = 0.0, 0.0, 0.0, 0.0

        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}/{args.epochs}", unit="batch", leave=False)
        batch_end_time = time.time()

        for batch in progress_bar:
            data_time = time.time() - batch_end_time
            compute_start = time.time()

            clean = batch["clean"].to(device, non_blocking=True)
            degraded = batch["degraded"].to(device, non_blocking=True)

            optimizer.zero_grad()

            with torch.amp.autocast(device.type, enabled=use_amp):
                recon_clean, mu_clean, logvar_clean = model(clean)
                recon_degraded, mu_degraded, logvar_degraded = model(degraded)
                loss, rc, rd, cons, kl = vae2_loss(
                    recon_clean, recon_degraded, clean, mu_clean, logvar_clean,
                    mu_degraded, logvar_degraded, kl_weight=args.kl_weight,
                    consistency_weight=args.consistency_weight,
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            if device.type == "cuda":
                torch.cuda.synchronize()
            compute_time = time.time() - compute_start

            running_loss += loss.item()
            running_rc += rc.item()
            running_rd += rd.item()
            running_cons += cons.item()
            running_kl += kl.item()
            global_step += 1

            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

            if args.log_every and global_step % args.log_every == 0:
                log(f"  step {global_step}: data_time={data_time:.3f}s compute_time={compute_time:.3f}s")

            if global_step % args.sample_every == 0:
                sample_path = os.path.join(samples_dir, f"step_{global_step:07d}.png")
                save_comparison_grid(model, fixed_batch, sample_path, device)
                log(f"  Saved sample grid: {sample_path}")

            batch_end_time = time.time()

        n_batches = len(dataloader)
        elapsed = time.time() - start_time
        log(f"[Epoch {epoch}/{args.epochs}] loss={running_loss / n_batches:.4f} "
            f"recon_clean={running_rc / n_batches:.4f} recon_degraded={running_rd / n_batches:.4f} "
            f"consistency={running_cons / n_batches:.4f} kl={running_kl / n_batches:.4f} "
            f"epoch_time={format_duration(time.time() - epoch_start)} "
            f"total_elapsed={format_duration(elapsed)}")

        if epoch % args.save_every == 0 or epoch == args.epochs:
            ckpt_path = os.path.join(checkpoints_dir, f"vae_domain_b_epoch{epoch:04d}.pt")
            torch.save({
                "epoch": epoch, "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "args": vars(args),
            }, ckpt_path)
            log(f"  Saved checkpoint: {ckpt_path}")

    log(f"Training complete. Total time: {format_duration(time.time() - start_time)}")
    logger.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_translation_net.py
"""
Train the latent translation network: the actual repair mechanism of this
method. Combines two training signals each step:

1. SUPERVISED (synthetic pairs, full ground truth available): encode a
   clean photo and its synthetically-degraded twin through frozen VAE2,
   translate the degraded latent with the mask, and directly supervise
   against the known clean latent with L1 loss. This teaches the network
   HOW to use the mask to repair damage.

2. ADVERSARIAL (real old photos, no ground truth): encode a real photo
   through frozen VAE1, translate it (no mask -- we don't know real
   damage locations), and train a discriminator to tell translated-real
   latents apart from genuine clean latents (from VAE2). The translation
   network is trained to fool it. This teaches the network to generalize
   the repair behavior learned from (1) to real-photo statistics, which
   is the actual domain-gap-closing step this whole method exists for.

Also includes an identity loss (translating an already-clean latent should
leave it roughly unchanged) which is a standard stabilizing trick borrowed
from CycleGAN-style unpaired translation training.

VAE1 and VAE2 are both loaded frozen from their own checkpoints and never
updated here -- only the translation network and discriminator train.

Usage:
    python train_translation_net.py \
        --vae1-checkpoint ./runs/vae_domain_a/checkpoints/vae_domain_a_epoch0050.pt \
        --vae2-checkpoint ./runs/vae_domain_b/checkpoints/vae_domain_b_epoch0050.pt \
        --real-photo-dir ./real_old_photos --clean-dir ./voc_data --masks-dir ./generated_masks \
        --epochs 50 --batch-size 8 --out-dir ./runs/translation_net
"""

import argparse
import os
import time
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, RandomSampler
import torchvision.utils as vutils
from tqdm import tqdm

from models.vae import DomainVAE
from models.translation_net import LatentTranslationNet, LatentDiscriminator
from data.real_photo_dataset import RealOldPhotoDataset, denormalize as denorm_a
from data.vae2_pair_dataset import VAE2PairDataset, denormalize as denorm_b


def format_duration(seconds):
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


class Logger:
    def __init__(self, log_path):
        os.makedirs(os.path.dirname(log_path) or ".", exist_ok=True)
        self._file = open(log_path, "a", encoding="utf-8")

    def log(self, msg):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] {msg}"
        print(line, flush=True)
        self._file.write(line + "\n")
        self._file.flush()

    def close(self):
        self._file.close()


def load_frozen_vae(checkpoint_path, device):
    """Loads a VAE1 or VAE2 checkpoint, reconstructing the architecture
    from the checkpoint's own saved args (same pattern as evaluate.py) so
    there's no risk of mismatching --latent-channels etc. by hand.
    Freezes all parameters -- this VAE is used only for encoding/decoding,
    never updated during translation-network training."""
    ckpt = torch.load(checkpoint_path, map_location=device)
    train_args = ckpt.get("args", {})
    model = DomainVAE(
        in_channels=3,
        n_downsample=train_args.get("n_downsample", 3),
        n_residual_blocks=train_args.get("n_residual_blocks", 4),
        latent_channels=train_args.get("latent_channels", 64),
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model, train_args


def save_sample_grid(vae1, vae2, translator, real_batch, synth_batch, out_path, device, max_images=4):
    """Two rows of comparisons:
      - synthetic: degraded | translated->decoded | clean target (has ground truth)
      - real: real damaged photo | translated->decoded (no ground truth, this
        is the actual real-world use case the whole method targets)
    """
    translator.eval()
    with torch.no_grad():
        n = min(max_images, synth_batch["clean"].shape[0])
        clean = synth_batch["clean"][:n].to(device)
        degraded = synth_batch["degraded"][:n].to(device)
        mask = synth_batch["mask"][:n].to(device)

        mu_degraded, _ = vae2.encoder(degraded)
        translated_synth = translator(mu_degraded, mask)
        decoded_synth = vae2.decoder(translated_synth)

        n_real = min(max_images, real_batch["image"].shape[0])
        real = real_batch["image"][:n_real].to(device)
        mu_real, _ = vae1.encoder(real)
        zero_mask = torch.zeros((n_real, 1, real.shape[-2], real.shape[-1]), device=device)
        translated_real = translator(mu_real, zero_mask)
        decoded_real = vae2.decoder(translated_real)

        row1 = torch.cat([denorm_b(degraded), denorm_b(decoded_synth), denorm_b(clean)], dim=0)
        row2 = torch.cat([denorm_a(real), denorm_a(decoded_real)], dim=0)

        vutils.save_image(row1, out_path.replace(".png", "_synthetic.png"), nrow=n)
        vutils.save_image(row2, out_path.replace(".png", "_real.png"), nrow=n_real)
    translator.train()


def main():
    parser = argparse.ArgumentParser(description="Train the latent translation network.")
    parser.add_argument("--vae1-checkpoint", type=str, required=True)
    parser.add_argument("--vae2-checkpoint", type=str, required=True)
    parser.add_argument("--real-photo-dir", type=str, required=True)
    parser.add_argument("--clean-dir", type=str, required=True)
    parser.add_argument("--masks-dir", type=str, required=True)
    parser.add_argument("--blend-mode", type=str, choices=["screen", "multiply"], default="screen")
    parser.add_argument("--out-dir", type=str, default="./runs/translation_net")
    parser.add_argument("--image-size", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--n-residual-blocks", type=int, default=6)
    parser.add_argument("--sup-weight", type=float, default=10.0,
                         help="weight on the supervised synthetic-pair loss")
    parser.add_argument("--identity-weight", type=float, default=1.0)
    parser.add_argument("--adv-weight", type=float, default=1.0)
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--save-every", type=int, default=5)
    parser.add_argument("--sample-every", type=int, default=200)
    parser.add_argument("--steps-per-epoch", type=int, default=100,
                         help="both the real-photo and synthetic-pair dataloaders are capped to this many "
                              "batches per epoch, so they stay aligned regardless of underlying dataset size")
    parser.add_argument("--log-every", type=int, default=20)
    parser.add_argument("--resume", type=str, default=None)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    checkpoints_dir = os.path.join(args.out_dir, "checkpoints")
    samples_dir = os.path.join(args.out_dir, "samples")
    os.makedirs(checkpoints_dir, exist_ok=True)
    os.makedirs(samples_dir, exist_ok=True)

    logger = Logger(os.path.join(args.out_dir, "train_log.txt"))
    log = logger.log

    device = torch.device(args.device)
    log(f"Using device: {device}")
    if device.type == "cuda":
        log(f"  GPU: {torch.cuda.get_device_name(device)}")

    log(f"Loading frozen VAE1 from {args.vae1_checkpoint}")
    vae1, vae1_args = load_frozen_vae(args.vae1_checkpoint, device)
    log(f"Loading frozen VAE2 from {args.vae2_checkpoint}")
    vae2, vae2_args = load_frozen_vae(args.vae2_checkpoint, device)

    latent_channels = vae1_args.get("latent_channels", 64)
    assert latent_channels == vae2_args.get("latent_channels", 64), \
        "VAE1 and VAE2 must share the same --latent-channels -- they were trained with different values."

    log("Building datasets...")
    real_dataset = RealOldPhotoDataset(args.real_photo_dir, image_size=args.image_size, augment=True)
    synth_dataset = VAE2PairDataset(args.clean_dir, args.masks_dir, image_size=args.image_size,
                                     augment=True, blend_mode=args.blend_mode)
    log(f"Real photos: {len(real_dataset)}. Synthetic pairs: {len(synth_dataset)} clean images, "
        f"{len(synth_dataset.mask_files)} masks.")

    real_sampler = RandomSampler(real_dataset, replacement=True,
                                  num_samples=args.steps_per_epoch * args.batch_size)
    synth_sampler = RandomSampler(synth_dataset, replacement=True,
                                   num_samples=args.steps_per_epoch * args.batch_size)
    real_loader = DataLoader(real_dataset, batch_size=args.batch_size, sampler=real_sampler,
                              num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                              persistent_workers=(args.num_workers > 0))
    synth_loader = DataLoader(synth_dataset, batch_size=args.batch_size, sampler=synth_sampler,
                               num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                               persistent_workers=(args.num_workers > 0))

    log("Fetching fixed sample batches for visualization...")
    fixed_real_batch = next(iter(real_loader))
    fixed_synth_batch = next(iter(synth_loader))
    log("Datasets ready.")

    translator = LatentTranslationNet(latent_channels=latent_channels,
                                       n_residual_blocks=args.n_residual_blocks).to(device)
    discriminator = LatentDiscriminator(latent_channels=latent_channels).to(device)

    opt_g = torch.optim.Adam(translator.parameters(), lr=args.lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=args.lr, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()

    start_epoch = 1
    global_step = 0
    if args.resume:
        log(f"Resuming from {args.resume}")
        ckpt = torch.load(args.resume, map_location=device)
        translator.load_state_dict(ckpt["translator_state_dict"])
        discriminator.load_state_dict(ckpt["discriminator_state_dict"])
        opt_g.load_state_dict(ckpt["opt_g_state_dict"])
        opt_d.load_state_dict(ckpt["opt_d_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt.get("global_step", 0)

    num_params = sum(p.numel() for p in translator.parameters())
    log(f"Translator has {num_params:,} parameters")
    log(f"Starting training: epochs {start_epoch}-{args.epochs}")

    start_time = time.time()

    for epoch in range(start_epoch, args.epochs + 1):
        epoch_start = time.time()
        running = {"sup": 0.0, "identity": 0.0, "adv_g": 0.0, "adv_d": 0.0}

        progress_bar = tqdm(zip(real_loader, synth_loader), total=args.steps_per_epoch,
                             desc=f"Epoch {epoch}/{args.epochs}", unit="batch", leave=False)
        batch_end_time = time.time()

        for real_batch, synth_batch in progress_bar:
            data_time = time.time() - batch_end_time
            compute_start = time.time()

            real = real_batch["image"].to(device, non_blocking=True)
            clean = synth_batch["clean"].to(device, non_blocking=True)
            degraded = synth_batch["degraded"].to(device, non_blocking=True)
            mask = synth_batch["mask"].to(device, non_blocking=True)

            with torch.no_grad():
                mu_clean, _ = vae2.encoder(clean)
                mu_degraded, _ = vae2.encoder(degraded)
                mu_real, _ = vae1.encoder(real)

            zero_mask_real = torch.zeros((mu_real.shape[0], 1, real.shape[-2], real.shape[-1]), device=device)

            # ---- Train discriminator ----
            opt_d.zero_grad()
            translated_real_detached = translator(mu_real, zero_mask_real).detach()
            d_real_out = discriminator(mu_clean)
            d_fake_out = discriminator(translated_real_detached)
            loss_d_real = bce(d_real_out, torch.ones_like(d_real_out))
            loss_d_fake = bce(d_fake_out, torch.zeros_like(d_fake_out))
            loss_d = 0.5 * (loss_d_real + loss_d_fake)
            loss_d.backward()
            opt_d.step()

            # ---- Train translator (generator) ----
            opt_g.zero_grad()

            translated_synth = translator(mu_degraded, mask)
            loss_sup = torch.nn.functional.l1_loss(translated_synth, mu_clean)

            zero_mask_clean = torch.zeros_like(mask)
            translated_identity = translator(mu_clean, zero_mask_clean)
            loss_identity = torch.nn.functional.l1_loss(translated_identity, mu_clean)

            translated_real = translator(mu_real, zero_mask_real)
            d_out_for_g = discriminator(translated_real)
            loss_adv_g = bce(d_out_for_g, torch.ones_like(d_out_for_g))

            loss_g = (args.sup_weight * loss_sup
                      + args.identity_weight * loss_identity
                      + args.adv_weight * loss_adv_g)
            loss_g.backward()
            opt_g.step()

            compute_time = time.time() - compute_start

            running["sup"] += loss_sup.item()
            running["identity"] += loss_identity.item()
            running["adv_g"] += loss_adv_g.item()
            running["adv_d"] += loss_d.item()
            global_step += 1

            progress_bar.set_postfix({"sup": f"{loss_sup.item():.4f}", "adv_d": f"{loss_d.item():.4f}"})

            if args.log_every and global_step % args.log_every == 0:
                log(f"  step {global_step}: data_time={data_time:.3f}s compute_time={compute_time:.3f}s")

            if global_step % args.sample_every == 0:
                sample_path = os.path.join(samples_dir, f"step_{global_step:07d}.png")
                save_sample_grid(vae1, vae2, translator, fixed_real_batch, fixed_synth_batch,
                                  sample_path, device)
                log(f"  Saved sample grids: {sample_path.replace('.png', '_synthetic.png')} / "
                    f"{sample_path.replace('.png', '_real.png')}")

            batch_end_time = time.time()

        elapsed = time.time() - start_time
        log(f"[Epoch {epoch}/{args.epochs}] "
            f"sup={running['sup'] / args.steps_per_epoch:.4f} "
            f"identity={running['identity'] / args.steps_per_epoch:.4f} "
            f"adv_g={running['adv_g'] / args.steps_per_epoch:.4f} "
            f"adv_d={running['adv_d'] / args.steps_per_epoch:.4f} "
            f"epoch_time={format_duration(time.time() - epoch_start)} "
            f"total_elapsed={format_duration(elapsed)}")

        if epoch % args.save_every == 0 or epoch == args.epochs:
            ckpt_path = os.path.join(checkpoints_dir, f"translation_net_epoch{epoch:04d}.pt")
            torch.save({
                "epoch": epoch, "global_step": global_step,
                "translator_state_dict": translator.state_dict(),
                "discriminator_state_dict": discriminator.state_dict(),
                "opt_g_state_dict": opt_g.state_dict(),
                "opt_d_state_dict": opt_d.state_dict(),
                "args": vars(args),
            }, ckpt_path)
            log(f"  Saved checkpoint: {ckpt_path}")

    log(f"Training complete. Total time: {format_duration(time.time() - start_time)}")
    logger.close()


if __name__ == "__main__":
    main()


## 4. Get your real old photos

Attach your uploaded dataset via **Add Data > Your Datasets**, then set `REAL_PHOTO_DIR` below. (See the project notes on using `kaggle datasets create` from your own machine if you haven't uploaded it yet.)

In [ ]:
REAL_PHOTO_DIR = '/kaggle/input/real-old-photos-vae1'  # <- adjust to your actual dataset slug

if os.path.isdir(REAL_PHOTO_DIR):
    n_images = len([f for f in os.listdir(os.path.join(REAL_PHOTO_DIR, 'images'))
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'Found {n_images} real photos at {REAL_PHOTO_DIR}')
else:
    print(f'{REAL_PHOTO_DIR} not found — attach your dataset via Add Data first.')


## 5. Get VOC2012 clean images — automatic, no manual download

In [ ]:
import torchvision.datasets as tvds

VOC_ROOT = '/kaggle/working/voc_data'
VOC_JPEG_DIR = os.path.join(VOC_ROOT, 'VOCdevkit', 'VOC2012', 'JPEGImages')

if os.path.isdir(VOC_JPEG_DIR) and len(os.listdir(VOC_JPEG_DIR)) > 0:
    print(f'VOC2012 already present at {VOC_JPEG_DIR}, skipping download.')
else:
    os.makedirs(VOC_ROOT, exist_ok=True)
    _ = tvds.VOCDetection(root=VOC_ROOT, year='2012', image_set='train', download=True)

num_images = len([f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')])
print(f'VOC2012 ready: {num_images} images at {VOC_JPEG_DIR}')


## 6. Generate damage masks

Clones FilmDamageSimulator and recreates `generate_synthetic_only.py`.

In [ ]:
if os.path.isdir('FilmDamageSimulator'):
    print('FilmDamageSimulator already cloned, skipping.')
else:
    !git clone --depth 1 https://github.com/daniela997/FilmDamageSimulator.git


In [ ]:
%%writefile FilmDamageSimulator/damage_generator/generate_synthetic_only.py
"""
Generate damage overlay masks using ONLY the pre-classified synthetic damage
patches in /synthetic/<type>/ (e.g. scratches, smut), without ever touching
the real scanned film frames in /scans/.

This bypasses damage_generator.py's default behaviour, which always loads
/scans/ and mixes real scanned artifact crops into the sampling pool even
when --synthetic is passed. Here, only the folder(s) you name are loaded,
and artifact count/size statistics are fit on those patches' own area
distribution instead of the real-scan-derived Gamma distributions.

Usage:
    python generate_synthetic_only.py --types scratches,smut --height 1024 --width 1024
    python generate_synthetic_only.py --types scratches --procedural-scratches
"""

import os
import argparse
import uuid
import random
import numpy as np
import pandas as pd
import cv2 as cv
import scipy.stats as stats
import skimage.transform as skimage_tf

from scans import load_images
from generate_masks import generate_perlin_noise_2d, increase_contrast, random_perlin_with_numpy, line_scratch


def sample_size_from_own_distribution(df, num_artifact):
    """Fit a Gamma distribution to this dataframe's OWN artifact areas
    (instead of a real-scan-derived one) and sample target sizes from it."""
    areas = df['Contour Area']
    gamma_param = stats.gamma.fit(areas, floc=0)
    shape, _, scale = gamma_param
    return np.random.gamma(shape, scale, num_artifact)


def sample_closest_in_area(df, target_areas):
    df = df.sample(frac=1).reset_index(drop=True)
    areas = df['Contour Area']
    indexes = []
    for target in target_areas:
        candidates = df.iloc[(areas - target).abs().argsort()[:15]].index.tolist()
        index = random.choice(candidates)
        indexes.append(index)
        areas = areas.drop(areas.index[[index]])
    picked = df.iloc[indexes].copy()
    picked['Target size'] = target_areas
    return picked


def build_mask(target_size, per_type_dfs, per_type_counts, rescale=True, verbose=False):
    rescale_factor = (target_size[0] / 2560 if target_size[0] <= target_size[1]
                       else target_size[1] / 2560) if rescale else 1.

    selected_frames = []
    for artifact_type, df in per_type_dfs.items():
        lo, hi = per_type_counts[artifact_type]
        num = int(np.random.randint(lo, hi + 1))
        if num == 0 or len(df) == 0:
            continue
        target_areas = sample_size_from_own_distribution(df, num)
        picked = sample_closest_in_area(df, target_areas)
        selected_frames.append(picked)
        if verbose:
            print(f"Selected {num} '{artifact_type}' artifacts")

    if not selected_frames:
        raise ValueError("No artifacts selected - check your --types and --min-count/--max-count")

    selected_artifacts_df = pd.concat(selected_frames, ignore_index=True)
    artifacts_num = len(selected_artifacts_df)

    mask_final = np.zeros(target_size).astype(np.uint8)
    perlin_noise = generate_perlin_noise_2d(target_size, (2, 2))
    normalised_noise = (perlin_noise - np.min(perlin_noise)) / np.ptp(perlin_noise)
    xs, ys = random_perlin_with_numpy(artifacts_num, normalised_noise)
    random_angles = np.random.randint(0, 360, size=artifacts_num)

    i = 0
    for _, artifact_row in selected_artifacts_df.iterrows():
        try:
            artifact = artifact_row['Artifact'].astype(np.uint8)
            random_scale = artifact_row['Target size'] / artifact_row['Contour Area']
            random_angle = random_angles[i]
            new_rescale_factor = rescale_factor * np.sqrt(random_scale)
            artifact = skimage_tf.rescale(artifact, round(new_rescale_factor, 2), anti_aliasing=True, preserve_range=True)
            artifact = skimage_tf.rotate(artifact, angle=random_angle, resize=True, preserve_range=True)
            artifact_w, artifact_h = artifact.shape[:2]

            x1 = xs[i] - artifact_w // 2
            x2 = x1 + artifact_w
            if x1 < 0:
                artifact = artifact[-x1:, :]; x1 = 0
            if x2 > target_size[0]:
                artifact = artifact[:-(x2 - target_size[0]), :]; x2 = target_size[0]

            y1 = ys[i] - artifact_h // 2
            y2 = y1 + artifact_h
            if y1 < 0:
                artifact = artifact[:, -y1:]; y1 = 0
            if y2 > target_size[1]:
                artifact = artifact[:, :-(y2 - target_size[1])]; y2 = target_size[1]

            mask_final[x1:x2, y1:y2] = np.where(
                artifact > mask_final[x1:x2, y1:y2], artifact, mask_final[x1:x2, y1:y2]
            )
            i += 1
        except Exception:
            i += 1
            continue

    mask_final = np.invert(mask_final.astype(np.uint8))
    binarised = ((mask_final > 240) * 255).astype(np.uint8)
    return mask_final.astype(np.uint8), binarised


def add_procedural_scratches(mask, height, width, verbose=False):
    """Blend in fully procedural (Perlin-noise-based) scratch lines.
    These require NO source images at all -- real or synthetic -- so they
    are always 'safe' to include without pulling in any scan data."""
    num_extra_scratch = int(np.random.gamma(6, 2, 1)[0])
    for _ in range(num_extra_scratch):
        length = np.random.randint(10, high=max(height, width), dtype=int)
        try:
            scratch = line_scratch(np.array(length))
            sw, sh = scratch.shape[:2]
            if sw >= width or sh >= height:
                continue
            x1 = np.random.randint(0, width - sw)
            y1 = np.random.randint(0, height - sh)
            region = mask[x1:x1 + sw, y1:y1 + sh]
            mask[x1:x1 + sw, y1:y1 + sh] = np.minimum(region, np.invert(scratch.astype(np.uint8)))
        except Exception:
            continue
    if verbose:
        print(f"Added {num_extra_scratch} procedural scratch lines")
    return mask


if __name__ == '__main__':
    parser = argparse.ArgumentParser(
        description='Generate damage masks from ONLY classified synthetic patches (no scanned frames).'
    )
    parser.add_argument('--types', type=str, default='scratches,smut',
                         help='comma-separated subfolder names under /synthetic/, '
                              'e.g. scratches,smut,dirt,dots,hair,hair-short,lint,sprinkles,spots,stain')
    parser.add_argument('--height', type=int, default=1024)
    parser.add_argument('--width', type=int, default=1024)
    parser.add_argument('--min-count', type=int, default=3, help='min number of artifacts per type')
    parser.add_argument('--max-count', type=int, default=15, help='max number of artifacts per type')
    parser.add_argument('--procedural-scratches', action='store_true',
                         help='also blend in fully procedural line scratches (no source image needed)')
    parser.add_argument('--n', type=int, default=1, help='how many masks to generate')
    parser.add_argument('--verbose', action='store_true')
    args = parser.parse_args()

    abs_path = os.path.abspath(os.path.dirname(__file__))
    synthetic_path = os.path.dirname(os.path.normpath(abs_path)) + '/synthetic/'
    out_dir = os.path.dirname(os.path.normpath(abs_path)) + '/generated/'
    os.makedirs(out_dir, exist_ok=True)

    types = [t.strip() for t in args.types.split(',') if t.strip()]

    per_type_dfs = {}
    for t in types:
        df = load_images(synthetic_path, t, verbose=args.verbose)
        df['Contour Area'] = df['Non-zero pixel area']
        per_type_dfs[t] = df
        print(f"Loaded {len(df)} '{t}' artifact patches from /synthetic/{t}/")

    per_type_counts = {t: (args.min_count, args.max_count) for t in types}

    for n in range(args.n):
        mask, binary_mask = build_mask(
            (args.height, args.width), per_type_dfs, per_type_counts, verbose=args.verbose
        )

        if args.procedural_scratches:
            mask = add_procedural_scratches(mask, args.height, args.width, verbose=args.verbose)
            binary_mask = ((mask > 240) * 255).astype(np.uint8)

        uid = str(uuid.uuid4())[:8]
        tag = "_".join(types)
        cv.imwrite(out_dir + f'mask_{tag}_{uid}.png', mask)
        cv.imwrite(out_dir + f'binarised_mask_{tag}_{uid}.png', binary_mask)
        print(f"[{n+1}/{args.n}] Saved mask_{tag}_{uid}.png")

    print(f"Done. Masks written to {out_dir}")


In [ ]:
MASKS_DIR = '/kaggle/working/generated_masks'
os.makedirs(MASKS_DIR, exist_ok=True)

TARGET_N_MASKS = 60
existing_masks = [f for f in os.listdir(MASKS_DIR) if f.startswith('mask_')]

if len(existing_masks) >= TARGET_N_MASKS:
    print(f'{len(existing_masks)} masks already present, skipping generation.')
else:
    os.chdir('/kaggle/working/FilmDamageSimulator/damage_generator')
    import shutil
    !python generate_synthetic_only.py --types scratches,smut \
        --height 256 --width 256 --min-count 3 --max-count 15 --n {TARGET_N_MASKS} --verbose
    os.chdir('/kaggle/working')
    src_dir = 'FilmDamageSimulator/generated'
    for fname in os.listdir(src_dir):
        shutil.copy(os.path.join(src_dir, fname), os.path.join(MASKS_DIR, fname))

n_masks = len([f for f in os.listdir(MASKS_DIR) if f.startswith('mask_')])
print(f'{n_masks} usable masks ready at {MASKS_DIR}')


## 7. Train VAE1 (domain A: real old photos)

Smoke test first (small, fast), then the real run. Skip to the real run directly if you already trained VAE1 in a previous session and just attached its checkpoint via Add Data.

In [ ]:
# Smoke test
!python train_vae_domain_a.py \
    --data-root "$REAL_PHOTO_DIR" \
    --epochs 3 --batch-size 8 --image-size 128 --num-workers 2 \
    --steps-per-epoch 10 --log-every 5 --sample-every 10 --save-every 1 \
    --amp --out-dir ./runs/vae_domain_a_smoke --device cuda


In [ ]:
# Real run — adjust --epochs once you've seen the smoke test succeed
# !python train_vae_domain_a.py \
#     --data-root "$REAL_PHOTO_DIR" \
#     --epochs 50 --batch-size 16 --image-size 256 --num-workers 2 \
#     --amp --out-dir ./runs/vae_domain_a --device cuda


## 8. Train VAE2 (domain B: clean photos + synthetic degradation)

Independent of VAE1 -- can run in parallel across sessions if you like.

In [ ]:
# Smoke test
!python train_vae_domain_b.py \
    --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
    --epochs 3 --batch-size 8 --image-size 128 --num-workers 2 \
    --steps-per-epoch 10 --log-every 5 --sample-every 10 --save-every 1 \
    --amp --out-dir ./runs/vae_domain_b_smoke --device cuda


In [ ]:
# Real run
# !python train_vae_domain_b.py \
#     --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
#     --epochs 50 --batch-size 16 --image-size 256 --num-workers 2 \
#     --amp --out-dir ./runs/vae_domain_b --device cuda


## 9. Train the latent translation network

Requires trained VAE1 and VAE2 checkpoints from steps 7-8 (both frozen here, only the translator + discriminator train). Point `--vae1-checkpoint` / `--vae2-checkpoint` at your actual latest checkpoint filenames — adjust the epoch number in the path below to match what you actually trained.

In [ ]:
VAE1_CKPT = './runs/vae_domain_a_smoke/checkpoints/vae_domain_a_epoch0003.pt'  # <- adjust
VAE2_CKPT = './runs/vae_domain_b_smoke/checkpoints/vae_domain_b_epoch0003.pt'  # <- adjust

!python train_translation_net.py \
    --vae1-checkpoint "$VAE1_CKPT" --vae2-checkpoint "$VAE2_CKPT" \
    --real-photo-dir "$REAL_PHOTO_DIR" --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
    --epochs 3 --batch-size 8 --image-size 128 --num-workers 2 \
    --steps-per-epoch 20 --log-every 5 --sample-every 20 --save-every 1 \
    --out-dir ./runs/translation_net_smoke --device cuda


In [ ]:
# Real run — point at your real VAE1/VAE2 checkpoints once trained
# !python train_translation_net.py \
#     --vae1-checkpoint ./runs/vae_domain_a/checkpoints/<latest>.pt \
#     --vae2-checkpoint ./runs/vae_domain_b/checkpoints/<latest>.pt \
#     --real-photo-dir "$REAL_PHOTO_DIR" --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
#     --epochs 50 --batch-size 16 --image-size 256 --num-workers 2 \
#     --out-dir ./runs/translation_net --device cuda


## 10. View translation network samples

Two grids: the synthetic branch (has ground truth: degraded | restored | clean) and the real branch (no ground truth: real damaged photo | restored — this is the actual real-world use case).

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

for suffix, title in [('_synthetic.png', 'Synthetic: degraded | restored | clean'),
                       ('_real.png', 'Real: damaged photo | restored')]:
    sample_files = sorted(glob.glob(f'runs/translation_net_smoke/samples/*{suffix}'))
    if sample_files:
        img = Image.open(sample_files[-1])
        plt.figure(figsize=(14, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title)
        plt.show()
    else:
        print(f'No {suffix} samples found yet.')


## Persistence notes

Click **Save Version** to preserve `/kaggle/working/` (including all checkpoints) as this version's Output. To continue training any of the three stages in a later session, attach your own previous Output as an input dataset and pass the copied checkpoint to `--resume`.

Check each stage's `train_log.txt` inside its `--out-dir` for a full timestamped history if you need to check progress after the fact.
